# Description
This script compares cest values of regions within network vs between network.


## Import Packages

In [1]:
import os
import glob
import numpy as np
import pandas as pd
#import network_fcon as fc
import scipy as sp
from scipy.stats import pearsonr
from scipy.stats import linregress
import seaborn as sns
import matplotlib.pyplot as plt
import re
import seaborn as sns
import statsmodels.formula.api as smf
from netneurotools import datasets, plotting
import nibabel as nib
from netneurotools.datasets import fetch_cammoun2012

### Set Variables and Paths

In [3]:
# Set Variables
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500' #'Schaefer2018_1000Parcels_17Networks'
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
nmaps = ["NMDA", "mGluR5", "GABA"]
figpath = '/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/figures/' + atlas

# Import Data
means_df = pd.read_csv(inpath + '/means_std_' + dataset + '_' + atlas + '.csv')


In [ ]:
# np.where - try this 

### NMAP correlations

In [4]:
print(means_df)
nmap_palette = {
    "NMDA": "Purples",   # Replace with actual nmap names and desired palettes
    "mGluR5": "Reds",
    "GABA": "Greens",
    "D2": "Oranges"
    # Add more mappings as needed
}

for nmap in nmaps:
    #means_df[nmap] = pd.to_numeric(means_df[nmap], errors='coerce')
    #means_df['CESTavg'] = pd.to_numeric(means_df['CESTavg'], errors='coerce')
    graph_df = means_df  #.dropna(subset=[nmap, 'CESTavg'])
    palette = nmap_palette.get(nmap) # Default to "Set1" if nmap not in dictionary
    plot = sns.lmplot(x=nmap, y='CESTavg', hue='hstatus', data=graph_df, palette=palette)
    plt.xlabel(nmap)
    plt.ylabel("CESTavg")
    plt.title('Linear Regression between ' + nmap + ' and CEST')
    
    slope, intercept, r_value, p_value, std_err = linregress(graph_df.loc[graph_df['hstatus'] == 'PSY', 'CESTavg'], graph_df.loc[graph_df['hstatus'] == 'PSY', nmap])
    plt.text(0.1, 0.8, f'PSY Group\nSlope: {slope:.2f}\nR^2: {r_value**2:.2f}\np: {p_value:.2f}', transform=plt.gca().transAxes)
    slope, intercept, r_value, p_value, std_err = linregress(graph_df.loc[graph_df['hstatus'] == 'HC', 'CESTavg'], graph_df.loc[graph_df['hstatus'] == 'HC', nmap])
    plt.text(0.4, 0.8, f'HC Group\nSlope: {slope:.2f}\nR^2: {r_value**2:.2f}\np: {p_value:.2f}', transform=plt.gca().transAxes)
    slope, intercept, r_value, p_value, std_err = linregress(graph_df['CESTavg'], graph_df[nmap])
    plt.text(0.7, 0.8, f'All Group\nSlope: {slope:.2f}\nR^2: {r_value**2:.2f}\np: {p_value:.2f}', transform=plt.gca().transAxes)

    plt.show() 


                         Parcel hstatus  CEST_avg      NMDA    mGluR5  \
0    caudalanteriorcingulate_5R      HC  0.140048 -0.026330  1.229935   
1    caudalanteriorcingulate_5R     PSY -0.011085 -0.026330  1.229935   
2                     cuneus_3R      HC  0.884299 -0.522157  0.053661   
3                     cuneus_3R     PSY  0.305783 -0.522157  0.053661   
4                     cuneus_5R      HC  1.411665  0.046650  0.220775   
..                          ...     ...       ...       ...       ...   
99           superiorfrontal_7R     PSY -0.455633  0.739121  1.834678   
100          superiorfrontal_8R      HC -0.857712 -0.331275 -0.129873   
101          superiorfrontal_8R     PSY -0.800876 -0.331275 -0.129873   
102          superiorfrontal_9R      HC -0.154002 -0.293822 -0.660154   
103          superiorfrontal_9R     PSY -0.372532 -0.293822 -0.660154   

         GABA  
0   -0.473436  
1   -0.473436  
2    0.654213  
3    0.654213  
4    1.635019  
..        ...  
99   1.1778

KeyError: "['CESTavg'] not in index"

In [45]:
if atlas == 'Schaefer2018_1000Parcels_17Networks':  
    schaefer_nifti = nib.load('/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/test_masks/Schaefer2018_1000Parcels_17Networks_order_FSLMNI152_1mm.nii.gz')
    schaefer_data = schaefer_nifti.get_fdata()
elif atlas == 'atl-Cammoun2012_res-500':
    print('yes')
"""
# CEST DATA
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'
figpath = '/Users/pecsok/projects/GluCEST-fMRI/glucest-rsfmri/fmri_pipeline/parcellated_pipeline/figures'
cestavg_df = pd.DataFrame(index=range(1, 1001), columns=['CESTavg'])
cestavg_df['CESTavg'] = np.nan  # Initialize all values to 0

for _, row in cestavg_df.iterrows():
    # Extract parcel ID number using regex
    match = re.search(r'(\d+)', row['parcel'])
    if match:
        parcel_id = int(match.group(1))
        # Add CESTavg data to corresponding row in the new DataFrame
        if 1 <= parcel_id <= 1000:  # Ensure the parcel ID is within range
            cestavg_df.at[parcel_id, 'CESTavg'] = row['CESTavg']
            
for i, row in cestavg_df.iterrows():
    parcel_id = i
    value = row['CESTavg']
print(cestavg_df)

    
parcel_values = pd.DataFrame({
    'parcel_id': range(1, 1001),  # Parcel IDs should range from 1 to 1000
    'value': cestavg_df['CESTavg'] # np.random.rand(1000)  # Replace with your actual values
})
print(parcel_values)


# Create a copy of the Schaefer atlas array to modify
new_schaefer_data = np.zeros(schaefer_data.shape)

# Loop through each parcel and assign the corresponding value from your DataFrame
for i, row in parcel_values.iterrows():
    parcel_id = row['parcel_id']
    value = row['value']
    
    # Assign the value to all voxels belonging to the current parcel ID
    new_schaefer_data[schaefer_data == parcel_id] = value

# Create a new NIfTI image with the updated data
new_nifti = nib.Nifti1Image(new_schaefer_data, affine=schaefer_nifti.affine, header=schaefer_nifti.header)

# Save the new NIfTI file
new_nifti.to_filename('updated_schaefer_atlas.nii.gz')
"""

      CESTavg
1         NaN
2         NaN
3         NaN
4         NaN
5         NaN
...       ...
996       NaN
997       NaN
998       NaN
999       NaN
1000      NaN

[1000 rows x 1 columns]
      parcel_id  value
1             1    NaN
2             2    NaN
3             3    NaN
4             4    NaN
5             5    NaN
...         ...    ...
996         996    NaN
997         997    NaN
998         998    NaN
999         999    NaN
1000       1000    NaN

[1000 rows x 2 columns]


{'scale033': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-033_deterministic.nii.gz', 'scale060': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-060_deterministic.nii.gz', 'scale125': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-125_deterministic.nii.gz', 'scale250': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-250_deterministic.nii.gz', 'scale500': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-500_deterministic.nii.gz', 'info': '/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_info.csv'}
/Users/pecsok/nnt-data/atl-cammoun2012/MNI152NLin2009aSym/atl-Cammoun2012_space-MNI152NLin2009aSym_res-500_deterministic.nii.gz


In [59]:
# Load in atlas 
if atlas == 'Schaefer2018_1000Parcels_17Networks':  
    schaefer_nifti = nib.load('/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/test_masks/Schaefer2018_1000Parcels_17Networks_order_FSLMNI152_1mm.nii.gz')
    schaefer_data = schaefer_nifti.get_fdata()
elif atlas == 'atl-Cammoun2012_res-500':
    cammoun = fetch_cammoun2012()
    niftipath = cammoun['scale500']
    info = pd.read_csv(cammoun['info'], sep=',')
    info = info[info['scale']== 'scale500']
    nifti = nib.load(niftipath)
    data = nifti.get_fdata()
    print(data.shape)
    print(info)
    
    

    

(197, 233, 189)
        id                    label hemisphere  structure     scale  \
909      1   lateralorbitofrontal_9          R     cortex  scale500   
910      2  lateralorbitofrontal_11          R     cortex  scale500   
911      3   lateralorbitofrontal_5          R     cortex  scale500   
912      4   lateralorbitofrontal_6          R     cortex  scale500   
913      5   lateralorbitofrontal_7          R     cortex  scale500   
...    ...                      ...        ...        ...       ...   
1919  1011                 pallidum          L  subcortex  scale500   
1920  1012            accumbensarea          L  subcortex  scale500   
1921  1013              hippocampus          L  subcortex  scale500   
1922  1014                 amygdala          L  subcortex  scale500   
1923  1015                brainstem          L  subcortex  scale500   

          yeo_7                von_economo  
909      limbic  primary/secondary sensory  
910      limbic  primary/secondary sensor

In [60]:
means_df = means_df[means_df['hstatus']=='HC']

In [61]:
# CEST DATA
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'

# Create df to store parcellated data
cestavg_df = pd.DataFrame(index=range(1, 1001), columns=['CEST_avg'])
cestavg_df['CEST_avg'] = np.nan  # Initialize all values to 0
#print(means_df)
#print(info)

# Read in means_df 
means_df['Number'] = None
# Loop through rows in means_df
for index, row in means_df.iterrows():
    parcelval = row['Parcel'][:-1]  # Get the parcel value without the last letter
    hemival = row['Parcel'][-1]     # Get just the last letter for the hemisphere
    # Initialize an empty list to collect matching ids
    matching_ids = []
    for j in range(len(info)):
        if info.iloc[j, info.columns.get_loc('label')] == parcelval and info.iloc[j, info.columns.get_loc('hemisphere')] == hemival:
            # Append the matching id to the list
            matching_ids.append(info.iloc[j, info.columns.get_loc('id')])
    # After looping through all info rows, assign the list of ids to 'Parcel number' column
    means_df.at[index, 'Number'] = ', '.join(map(str, matching_ids))  # Join as a string

#print(means_df)


NewData = pd.DataFrame({
    'Parcel_id': range(1, 1016),  # Numbers 1 to 1015
    'Value': 0  # Initialize the 'Value' column with None
})

for _, means_row in means_df.iterrows():
    # Get the value from means_df's 'number' column
    parcel_val = int(means_row['Number'])
    #print(parcel_val.type)
    cest_val = means_row['CEST_avg']
    #print(cest_val)
    # Loop through NewData rows
    for idx, newdata_row in NewData.iterrows():
        if newdata_row['Parcel_id'] == parcel_val:
            NewData.at[idx, 'Value'] = cest_val       
print(NewData)

"""
for i, row in cestavg_df.iterrows():
    parcel_id = i
    value = row['CEST_avg']
print(cestavg_df)

    
parcel_values = pd.DataFrame({
    'parcel_id': range(1, 1001),  # Parcel IDs should range from 1 to 1000
    'value': cestavg_df['CESTavg'] # np.random.rand(1000)  # Replace with your actual values
})
print(parcel_values)
"""

# Create a copy of the Schaefer atlas array to modify
new_data = np.zeros(data.shape)
print(new_data.shape)


# Loop through each parcel and assign the corresponding value from your DataFrame
for i, row in NewData.iterrows():
    parcel_id = row['Parcel_id']
    value = row['Value']
    
    # Assign the value to all voxels belonging to the current parcel ID
    new_data[data == parcel_id] = value

# Create a new NIfTI image with the updated data
new_nifti = nib.Nifti1Image(new_data, affine=nifti.affine, header=nifti.header)

# Save the new NIfTI file
new_nifti.to_filename('CEST' + atlas + '.nii.gz')


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_97972/3537883197.py:42: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1400475790386027' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  NewData.at[idx, 'Value'] = cest_val


      Parcel_id  Value
0             1    0.0
1             2    0.0
2             3    0.0
3             4    0.0
4             5    0.0
...         ...    ...
1010       1011    0.0
1011       1012    0.0
1012       1013    0.0
1013       1014    0.0
1014       1015    0.0

[1015 rows x 2 columns]
(197, 233, 189)


In [22]:
# CEST DATA
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'

# Create df to store parcellated data
cestavg_df = pd.DataFrame(index=range(1, 1001), columns=['CEST_avg'])
cestavg_df['CEST_avg'] = np.nan  # Initialize all values to 0
#print(means_df)
print(info)

# Read in means_df 
means_df['Number'] = None
# Loop through rows in means_df
for index, row in means_df.iterrows():
    parcelval = row['Parcel'][:-1]  # Get the parcel value without the last letter
    hemival = row['Parcel'][-1]     # Get just the last letter for the hemisphere

    # Initialize an empty list to collect matching ids
    matching_ids = []
    for j in range(len(info)):
        if info.iloc[j, info.columns.get_loc('label')] == parcelval and info.iloc[j, info.columns.get_loc('hemisphere')] == hemival:
            # Append the matching id to the list
            matching_ids.append(info.iloc[j, info.columns.get_loc('id')])
    
    # After looping through all info rows, assign the list of ids to 'Parcel number' column
    means_df.at[index, 'Number'] = ', '.join(map(str, matching_ids))  # Join as a string

print(means_df)

"""
# Loop through Parcel column rows
for _, row in means_df.iterrows():
    parcelval = means_df.loc(col='Parcel', row=row).values but remove the last letter
    hemival = means_df.loc(col='Parcel', row=row).values but just the last letter
    # Now loop through info df.
    for i in range(len(info)):
    # If info(label) == parcelval and info['hemisphere'] == hemival
          means_df.loc(col='Parcel number', row=row) = info.loc(row=j, col='id'))

# Load in Parcel labels.

for _, row in means_df.iterrows():
    # Extract parcel ID number using regex
    match = re.search(r'(\d+)', row['Parcel'])
    if match:
        parcel_id = int(match.group(1))
        print(parcel_id)
        
        # Add CESTavg data to corresponding row in the new DataFrame
        if 1 <= parcel_id <= 1000:  # Ensure the parcel ID is within range
            cestavg_df.at[parcel_id, 'CEST_avg'] = row['CEST_avg']
"""

for i, row in cestavg_df.iterrows():
    parcel_id = i
    value = row['CEST_avg']
print(cestavg_df)

    
parcel_values = pd.DataFrame({
    'parcel_id': range(1, 1001),  # Parcel IDs should range from 1 to 1000
    'value': cestavg_df['CESTavg'] # np.random.rand(1000)  # Replace with your actual values
})
print(parcel_values)


# Create a copy of the Schaefer atlas array to modify
new_schaefer_data = np.zeros(schaefer_data.shape)

# Loop through each parcel and assign the corresponding value from your DataFrame
for i, row in parcel_values.iterrows():
    parcel_id = row['parcel_id']
    value = row['value']
    
    # Assign the value to all voxels belonging to the current parcel ID
    new_schaefer_data[schaefer_data == parcel_id] = value

# Create a new NIfTI image with the updated data
new_nifti = nib.Nifti1Image(new_schaefer_data, affine=schaefer_nifti.affine, header=schaefer_nifti.header)

# Save the new NIfTI file
new_nifti.to_filename('updated_schaefer_atlas.nii.gz')


        id                    label hemisphere  structure     scale  \
909      1   lateralorbitofrontal_9          R     cortex  scale500   
910      2  lateralorbitofrontal_11          R     cortex  scale500   
911      3   lateralorbitofrontal_5          R     cortex  scale500   
912      4   lateralorbitofrontal_6          R     cortex  scale500   
913      5   lateralorbitofrontal_7          R     cortex  scale500   
...    ...                      ...        ...        ...       ...   
1919  1011                 pallidum          L  subcortex  scale500   
1920  1012            accumbensarea          L  subcortex  scale500   
1921  1013              hippocampus          L  subcortex  scale500   
1922  1014                 amygdala          L  subcortex  scale500   
1923  1015                brainstem          L  subcortex  scale500   

          yeo_7                von_economo  
909      limbic  primary/secondary sensory  
910      limbic  primary/secondary sensory  
911      lim

KeyError: 'CESTavg'